<a href="https://colab.research.google.com/github/r80gustavo-coder/Gustavo/blob/main/Gerador_de_Pacote_Gemini_Veo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path
import zipfile
import json
import shutil
import os

base = Path("/mnt/data/gemini-veo-windows-package")
zip_path = Path("/mnt/data/gemini-veo-windows-package.zip")

# Limpa o diretório base e o zip existente
if base.exists():
    shutil.rmtree(base)
if zip_path.exists():
    os.remove(zip_path)

base.mkdir(parents=True, exist_ok=True)

# README
readme = """Gustavo — Projeto Gemini + Veo (Windows)
========================================

O que este pacote faz
- Gera uma imagem usando a API Gemini (Nano Banana).
- Submete essa imagem para gerar um vídeo com Veo 3.1.
- Faz polling até o vídeo ficar pronto e baixa o arquivo.
- Pergunta o prompt toda vez que rodar (modo interativo).

ATENÇÃO IMPORTANTE
- Este projeto NÃO inclui sua chave de API. Você deve copiar sua chave GEMINI_API_KEY para um arquivo chamado `.env` na mesma pasta.
- Exemplo de `.env`:
  GEMINI_API_KEY=AIzaSy...
- NÃO compartilhe sua chave publicamente.

Requisitos
- Windows
- Node.js (v18+ recomendado). Baixe em https://nodejs.org (instale LTS).
- Internet ativa.

Como usar (passo-a-passo simples)
1. Extraia o ZIP para C:\\meu-projeto-gemini (ou onde preferir).
2. Abra o Explorador e clique duas vezes em `start.bat`.
  - O script vai instalar dependências (npm install) e rodar o programa.
3. Quando o programa pedir, digite o prompt para gerar a imagem e depois o prompt para o vídeo.
4. Aguarde (o processo faz polling). Os arquivos gerados:
  - imagem.png  (imagem criada)
  - video.mp4   (vídeo baixado, se o Veo estiver disponível para sua conta)

O que pode dar errado e soluções rápidas
- Erro "permission denied" ou "model not found": sua conta pode não ter acesso ao Veo 3.1 (paywalled/preview). Nesse caso o script irá pelo menos gerar a imagem e avisar.
- Erro de autenticação: confirme se o arquivo `.env` existe e contém GEMINI_API_KEY com a chave correta.
- Se algo falhar, cole aqui a mensagem de erro e eu te ajudo.

Segurança
- Mantenha a chave `.env` privada. Não faça upload dela para repositórios públicos.

Boa sorte — quando rodar e se aparecer algum erro, me manda a mensagem e eu te ajudo a corrigir.
"""
(base / "README.txt").write_text(readme, encoding="utf-8")

# .env.example
env_example = """# Copie este arquivo para `.env` e substitua a chave abaixo:
GEMINI_API_KEY=COLE_SUA_CHAVE_AQUI
"""
(base / ".env.example").write_text(env_example, encoding="utf-8")

# start.bat
start_bat = r"""@echo off
REM Windows start script: instala dependências e executa
echo Instalando dependencias...
npm install
if %errorlevel% neq 0 (
  echo Erro ao instalar dependencias. Verifique sua conexao e permissoes.
  pause
  exit /b 1
)
echo Rodando o app...
node app.js
pause
"""
(base / "start.bat").write_text(start_bat, encoding="utf-8")

# package.json
pkg = {
  "name": "gemini-veo-windows-package",
  "version": "1.0.0",
  "description": "Demo project to generate image (Gemini) and request Veo 3.1 video. Interactive prompts.",
  "main": "app.js",
  "scripts": {
    "start": "node app.js"
  },
  "dependencies": {
    "axios": "^1.5.0",
    "dotenv": "^16.3.0",
    "@google/genai": "^1.0.0"
  },
  "author": "Generated for Gustavo",
  "license": "MIT"
}
(base / "package.json").write_text(json.dumps(pkg, indent=2), encoding="utf-8")

# helpers.js - small utilities
helpers_js = """const fs = require('fs');
function saveBase64ToFile(base64, filename) {
  const buf = Buffer.from(base64, 'base64');
  fs.writeFileSync(filename, buf);
}
module.exports = { saveBase64ToFile };
"""
(base / "helpers.js").write_text(helpers_js, encoding="utf-8")

# app.js - main application
app_js = r"""/**
 * app.js
 * Interactive Windows-friendly script:
 * - Reads GEMINI_API_KEY from .env
 * - Asks for image prompt and video prompt
 * - Tries to generate image via Gemini
 * - Tries to submit Veo 3.1 job and poll until ready
 *
 * Notes:
 * - The Veo endpoints/fields may be in preview. If your account lacks access, the script will
 * show the error and still keep the generated image.
 * - This script uses axios + direct REST calls so you can adapt quickly.
 */

require('dotenv').config();
const fs = require('fs');
const axios = require('axios');
const readline = require('readline');
const { saveBase64ToFile } = require('./helpers');

const API_KEY = process.env.GEMINI_API_KEY;
if (!API_KEY || API_KEY.includes('COLE_SUA')) {
  console.error('ERRO: coloque sua chave GEMINI_API_KEY em um arquivo .env (veja .env.example).');
  process.exit(1);
}

function ask(question) {
  const rl = readline.createInterface({
    input: process.stdin,
    output: process.stdout
  });
  return new Promise(res => rl.question(question, ans => { rl.close(); res(ans); }));
}

async function gerarImagem(prompt) {
  console.log('📷 Solicitando imagem ao Gemini (Nano Banana)...');

  // Endpoint de imagens (padrão publicamente documentado em alguns exemplos)
  const url = 'https://generativelanguage.googleapis.com/v1/images:generate';

  // Corpo simples: ajuste conforme sua disponibilidade/necessidade
  const body = {
    model: 'image-bison-001', // nome comum em docs; se sua conta usar outro nome, ajuste
    prompt: {
      text: prompt
    },
    imageFormat: 'PNG',
    size: '1024x1024'
  };

  try {
    const resp = await axios.post(url, body, {
      headers: {
        'Authorization': `Bearer ${API_KEY}`,
        'Content-Type': 'application/json'
      },
      timeout: 120000
    });
    // A forma exata do retorno varia; tentamos detectar campos comuns
    const data = resp.data || {};
    // Tenta várias locações possíveis para o conteúdo base64
    const base64 = data?.image?.imageBytes || data?.images?.[0]?.imageBytes || data?.data?.[0]?.b64_json || data?.image_base64 || data?.base64;
    if (!base64) {
      console.warn('⚠️ Resposta recebida, mas não foi possível localizar imagem em base64 automaticamente.');
      console.log('Resposta completa da API:', JSON.stringify(data).slice(0,1000));
      throw new Error('Formato de resposta inesperado. Veja o JSON acima.');
    }
    saveBase64ToFile(base64, 'imagem.png');
    console.log('✅ Imagem salva em imagem.png');
    return 'imagem.png';
  } catch (err) {
    console.error('Erro ao gerar imagem:', err.response?.data || err.message);
    throw err;
  }
}

async function uploadImageForVeo(imagePath) {
  console.log('🔁 Preparando imagem para envio ao Veo...');
  // Em muitos fluxos, Veo aceita URL pública ou base64 dentro do payload.
  // Aqui usamos base64 e enviamos diretamente (se suportado).
  const b64 = fs.readFileSync(imagePath).toString('base64');
  return b64;
}

async function criarJobVeo(imageBase64, promptVideo) {
  console.log('🎬 Solicitando geração de vídeo (Veo 3.1)...');

  // NOTE: Este endpoint e formato são ilustrativos. Se sua conta retornar "model not found",
  // isso indica que Veo 3.1 não está disponível para sua chave/conta.
  const url = 'https://generativelanguage.googleapis.com/v1beta/video:generate';
  const body = {
    model: 'veo-3.1-i2v',
    prompt: promptVideo,
    input_image_base64: imageBase64,
    // Parâmetros opcionais: duration, fps, resolution
    duration_seconds: 6
  };

  try {
    const resp = await axios.post(url, body, {
      headers: {
        'Authorization': `Bearer ${API_KEY}`,
        'Content-Type': 'application/json'
      },
      timeout: 120000
    });
    // Espera retorno com operation id
    const op = resp.data?.name || resp.data?.operationId || resp.data?.operation_id;
    if (!op) {
      console.warn('Resposta do Veo não incluiu operation id:', JSON.stringify(resp.data).slice(0,1000));
      throw new Error('Operation id não encontrado na resposta.');
    }
    console.log('✅ Job criado. Operation id:', op);
    return op;
  } catch (err) {
    console.error('Erro ao criar job Veo:', err.response?.data || err.message);
    throw err;
  }
}

async function pollOperation(operationName, interval = 5000, timeoutMs = 10 * 60 * 1000) {
  console.log('⏳ Iniciando polling da operação...');
  const start = Date.now();
  const base = 'https://generativelanguage.googleapis.com/v1';
  const url = `${base}/${operationName.replace(/^\\//, '')}`;

  while (true) {
    if (Date.now() - start > timeoutMs) {
      throw new Error('Tempo de espera excedido ao aguardar operação.');
    }
    try {
      const resp = await axios.get(url, {
        headers: { 'Authorization': `Bearer ${API_KEY}` },
        timeout: 60000
      });
      const data = resp.data || {};
      if (data.done) {
        console.log('✅ Operação concluída.');
        // tenta obter a URL do vídeo (variações de campo)
        const videoUrl = data?.response?.video_url || data?.result?.videoUrl || data?.response?.output?.[0]?.uri || data?.output_url;
        return { data, videoUrl };
      } else {
        console.log('Ainda processando... aguardando', interval/1000, 's');
      }
    } catch (err) {
      console.error('Erro ao checar status da operação:', err.response?.data || err.message);
    }
    await new Promise(r => setTimeout(r, interval));
  }
}

async function downloadToFile(url, filename) {
  console.log('⬇️ Baixando', url);
  const resp = await axios.get(url, { responseType: 'stream' });
  const writer = fs.createWriteStream(filename);
  resp.data.pipe(writer);
  return new Promise((resolve, reject) => {
    writer.on('finish', () => resolve());
    writer.on('error', reject);
  });
}

(async () => {
  try {
    const promptImg = await ask('Digite o prompt para a IMAGEM: ');
    const imgPath = await gerarImagem(promptImg);

    // Prepare image
    const imgB64 = await uploadImageForVeo(imgPath);

    // Ask video prompt
    const promptVid = await ask('Digite o prompt para o VIDEO (curto): ');
    let operationId;
    try {
      operationId = await criarJobVeo(imgB64, promptVid);
    } catch (err) {
      console.warn('Não foi possível criar job Veo — talvez sua conta não tenha acesso. Processo finalizado com a imagem apenas.');
      process.exit(0);
    }

    // Poll
    const { data, videoUrl } = await pollOperation(operationId);
    if (videoUrl) {
      await downloadToFile(videoUrl, 'video.mp4');
      console.log('✅ Vídeo salvo em video.mp4');
    } else if (data?.response?.output?.[0]?.data?.b64_video) {
      // fallback: base64 video embed
      const b64 = data.response.output[0].data.b64_video;
      fs.writeFileSync('video.mp4', Buffer.from(b64, 'base64'));
      console.log('✅ Vídeo salvo em video.mp4 (via base64 no payload)');
    } else {
      console.warn('Operação concluída mas não foi possível localizar a URL ou base64 do vídeo. Veja o JSON completo abaixo:');
      console.log(JSON.stringify(data, null, 2));
    }

    console.log('🏁 Processo finalizado.');
  } catch (err) {
    console.error('Erro geral:', err.message || err);
    process.exit(1);
  }
})();
"""
(base / "app.js").write_text(app_js, encoding="utf-8")

# LICENSE
(base / "LICENSE").write_text("MIT License\n\nGenerated package for Gustavo.", encoding="utf-8")

# Create zip
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in base.rglob("*"):
        z.write(p, arcname=str(p.relative_to(base)))

str(zip_path)

'/mnt/data/gemini-veo-windows-package.zip'